# vLLM and Qdrant: hands-on

Plain HTTP calls to the two services from `docker-compose.yml`, so you can see the raw APIs.
Start them first: `docker compose up -d`.

- **vLLM** serves an OpenAI-compatible API on `:8000`
- **Qdrant** serves a REST API on `:6333` (dashboard: http://localhost:6333/dashboard)

In [1]:
import json

import httpx

from rag.config import load_config

cfg = load_config()
VLLM = cfg.llm.base_url   # http://localhost:8000/v1
MODEL = cfg.llm.model     # "llm" = --served-model-name
QDRANT = cfg.qdrant.url   # http://localhost:6333

## vLLM

### Which model is loaded
`GET /v1/models` — `id` is the name clients use, `root` is the Hugging Face repo behind it.

### Chat completion
`POST /v1/chat/completions` — same request shape as OpenAI. The model applies its chat template to `messages`,
generates up to `max_tokens`, and reports token counts in `usage`. `temperature=0` makes the output (nearly) deterministic.

In [2]:
def chat(prompt: str, **params) -> dict:
    body = {"model": MODEL, "messages": [{"role": "user", "content": prompt}], "max_tokens": 64, "temperature": 0, **params}
    return httpx.post(f"{VLLM}/chat/completions", json=body, timeout=60).json()

resp = chat("What is the capital of France? Answer in one word.")
print(resp["choices"][0]["message"]["content"])
print(resp["usage"])

Paris
{'prompt_tokens': 41, 'total_tokens': 43, 'completion_tokens': 2, 'prompt_tokens_details': None}


In [3]:
resp = chat("Какая столица Франции? Ответь одним словом.")
print(resp["choices"][0]["message"]["content"])
print(resp["usage"])

Париж
{'prompt_tokens': 45, 'total_tokens': 49, 'completion_tokens': 4, 'prompt_tokens_details': None}


### System prompt
A `system` message sets the rules for the conversation. This is how the RAG prompt (Phase 6) will tell the model
to answer only from the provided context.

In [4]:
body = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": "Answer only from the context. If the answer is not there, say 'not found'.\n\nContext: [1] Nikola Tesla died on 7 January 1943."},
        {"role": "user", "content": "When did Tesla die? And where was he born?"},
    ],
    "max_tokens": 64,
    "temperature": 0,
}
print(httpx.post(f"{VLLM}/chat/completions", json=body, timeout=60).json()["choices"][0]["message"]["content"])

Tesla died on 7 January 1943. The context does not provide information about where Tesla was born.


### Streaming
With `"stream": true` the server sends SSE (Server-Sent Events): one `data: {...}` line per chunk of tokens,
each holding a `delta`, then `data: [DONE]`. This is what makes time-to-first-token short.

In [5]:
body = {"model": MODEL, "messages": [{"role": "user", "content": "Расскажи в двух предложениях, что такое RAG."}],
        "max_tokens": 128, "temperature": 0, "stream": True}

with httpx.stream("POST", f"{VLLM}/chat/completions", json=body, timeout=60) as r:
    for line in r.iter_lines():
        if not line.startswith("data: "):
            continue
        data = line.removeprefix("data: ")
        if data == "[DONE]":
            break
        print(json.loads(data)["choices"][0]["delta"].get("content", ""), end="", flush=True)

RAG — это методика или архитектура для моделирования и использования релевантных контекстных данных, где "R" означает Retrieval (извлечение), "A" — Augmentation (расширение), а "G" — Generation (генерация). Этот подход позволяет моделям машинного обучения использовать внешние данные для улучшения точности и релевантности ответов.

## Qdrant

### Server status
`GET /collections` lists collections. Fresh install → empty list.

In [6]:
print(httpx.get(QDRANT).json())               # version info
httpx.get(f"{QDRANT}/collections").json()

{'title': 'qdrant - vector search engine', 'version': '1.19.1', 'commit': '6ab21cac18ebb6f4ae29102c7f8f5cc11affd5de'}


{'result': {'collections': [{'name': 'chunks'}]},
 'status': 'ok',
 'time': 4.14e-06}

### Create a collection
A collection is like a table. Each point has an `id`, one or more **named vectors**, and a JSON **payload**.
Here: one named vector `dense` of size 4 with cosine distance (the real one will be 1024-dim bge-m3 vectors).
Creating a collection that already exists is an error, so we drop it first.

In [7]:
COL = f"{QDRANT}/collections/demo"
httpx.delete(COL)
httpx.put(COL, json={"vectors": {"dense": {"size": 4, "distance": "Cosine"}}}).json()

{'result': True, 'status': 'ok', 'time': 0.051167667}

### Insert points
`wait=true` returns only after the points are indexed and searchable.
The `acl` payload field mimics the permission groups the real index will store per chunk.

In [8]:
points = [
    {"id": 1, "vector": {"dense": [0.9, 0.1, 0.0, 0.0]}, "payload": {"text": "Tesla died in 1943", "acl": ["all-employees"]}},
    {"id": 2, "vector": {"dense": [0.8, 0.2, 0.1, 0.0]}, "payload": {"text": "Tesla was born in Smiljan", "acl": ["hr-team"]}},
    {"id": 3, "vector": {"dense": [0.0, 0.1, 0.9, 0.3]}, "payload": {"text": "The Rhine flows into the North Sea", "acl": ["all-employees"]}},
]
httpx.put(f"{COL}/points", params={"wait": "true"}, json={"points": points}).json()

{'result': {'operation_id': 1, 'status': 'completed'},
 'status': 'ok',
 'time': 0.00102971}

### Vector search
`POST /points/query` returns the nearest points by cosine similarity (`score`, higher = closer).

In [9]:
query = [1.0, 0.1, 0.0, 0.0]   # "about Tesla"

def search(**extra):
    body = {"query": query, "using": "dense", "limit": 3, "with_payload": True, **extra}
    for p in httpx.post(f"{COL}/points/query", json=body).json()["result"]["points"]:
        print(f"{p['score']:.3f}  {p['payload']}")

search()

1.000  {'text': 'Tesla died in 1943', 'acl': ['all-employees']}
0.982  {'text': 'Tesla was born in Smiljan', 'acl': ['hr-team']}
0.010  {'text': 'The Rhine flows into the North Sea', 'acl': ['all-employees']}


### Search with an ACL filter
The filter is applied **inside** the search, not after it, so `limit` results are all ones the user may see.
`match.any` = the point's `acl` shares at least one group with the user's groups.
A point without an `acl` field never matches — that's the fail-closed behavior the plan relies on.

In [10]:
alice_groups = ["all-employees"]
search(filter={"must": [{"key": "acl", "match": {"any": alice_groups}}]})

1.000  {'text': 'Tesla died in 1943', 'acl': ['all-employees']}
0.010  {'text': 'The Rhine flows into the North Sea', 'acl': ['all-employees']}


### Update payload without touching the vector
This is the cheap path for an ACL-only change: no re-embedding.

In [11]:
httpx.post(f"{COL}/points/payload", params={"wait": "true"}, json={"payload": {"acl": ["all-employees"]}, "points": [2]}).json()
search(filter={"must": [{"key": "acl", "match": {"any": alice_groups}}]})

1.000  {'text': 'Tesla died in 1943', 'acl': ['all-employees']}
0.982  {'text': 'Tesla was born in Smiljan', 'acl': ['all-employees']}
0.010  {'text': 'The Rhine flows into the North Sea', 'acl': ['all-employees']}


### Clean up

In [12]:
httpx.delete(COL).json()

{'result': True, 'status': 'ok', 'time': 0.003045791}

## Offline eval on the golden set
Same as `rag eval`: Recall@5/10, MRR, NDCG@10 per stage and language setting, plus the gate calibration table.
Needs Qdrant with the corpus indexed (`rag ingest`) and the embedder/reranker models (no vLLM).
`LIMIT` = answerable questions per language; `None` runs all 1190 per language (slow).

In [13]:
import os
from datetime import date
from pathlib import Path

import yaml
from qdrant_client import QdrantClient

from rag.eval.runner import Evaluator, format_report, sample
from rag.search.rerank import CrossEncoderReranker
from rag.search.retrieval import Retriever
from rag.storage.encoders import BgeM3Encoder, Bm25Encoder
from rag.storage.index import Index

# the notebook starts in tests/; config.yaml paths (models.embedder) are relative to the repo root
ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
os.chdir(ROOT)
LIMIT = 100

dense = BgeM3Encoder(cfg.models.embedder)
r = cfg.retrieval
retriever = Retriever(Index(QdrantClient(cfg.qdrant.url), cfg.qdrant.collection, dense.dim), dense, Bm25Encoder(),
                      r.candidates, r.rrf_k, r.rrf_weights)
evaluator = Evaluator(cfg, retriever, CrossEncoderReranker(cfg.models.reranker),
                      yaml.safe_load((ROOT / "data/users.yaml").read_text()), date.today())
golden = [json.loads(line) for line in (ROOT / "data/golden.jsonl").read_text().splitlines()]

/home/alex/Courses/QA/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 2646.41it/s]


In [14]:
# questions = sample(golden, LIMIT)
# retrieval, gate_scores = evaluator.retrieval(questions)
# gate = evaluator.gate(questions, gate_scores)
# print(format_report(retrieval, gate, cfg.rerank.gate_threshold))

### Regression test
Fails if Recall@5 drops below the floor on 100 questions per language (see `tests/integration/test_eval_regression.py`).

In [15]:
# !cd {ROOT} && pytest tests/integration/test_eval_regression.py -q

## Groundedness check
Answers the golden questions tagged `generation` (30 EN + 30 RU) with the real pipeline, so it needs **vLLM running** on top of
Qdrant with the corpus indexed. Each row of `tests/groundedness.jsonl` has what a judge needs: the question, the reference answers,
the exact prompt the LLM saw (question + numbered context passages), the answer and its citations. A judge scores each answered row:

- **faithful**: every claim in the answer is supported by the passages it cites
- **correct**: the answer agrees with the reference answers
- **citations**: each `[n]` points to a passage that contains the claim it is attached to

Same as `rag eval-generation`, but with a `GEN_LIMIT` for quick runs. Rows the gate blocked have `status: not_found` and `prompt: null`.

In [16]:
from rag.eval.generation import RecordingLLM, run_generation
from rag.generation.llm import OpenAICompatLLM
from rag.indexing.chunking import make_token_counter
from rag.serving.conversations import ConversationStore
from rag.serving.pipeline import Pipeline

GEN_LIMIT = None  # None = all 60 questions
OUT = ROOT / "tests/groundedness.jsonl"

llm = RecordingLLM(OpenAICompatLLM(cfg.llm.base_url, cfg.llm.model))
pipeline = Pipeline(cfg, evaluator.retriever, evaluator.reranker, llm,
                    make_token_counter(cfg.llm.tokenizer), ConversationStore())
retriever.index.require_embedder(dense.name)

gen_questions = [g for g in golden if "generation" in g["tags"]][:GEN_LIMIT]
rows = run_generation(pipeline, llm, gen_questions, evaluator.users)
OUT.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows))
statuses = [row["status"] for row in rows]
print(f"{len(rows)} questions -> {OUT}: " + ", ".join(f"{s} {statuses.count(s)}" for s in sorted(set(statuses))))

2026-10-16 11:08:07.562 | INFO     | rag.serving.pipeline:ask:75 - {"user": "root", "conversation_id": "a18b079b09b643ba9f5a8e4cdc71ace2", "query": "What actor did sign language for the National Anthem at Superbowl 50?", "status": "answered", "chunks": ["art00-en-c003", "art00-ru-c003", "art34-en-c004", "art31-en-c004", "art22-en-c002", "art24-en-c004", "art00-en-c002", "art34-en-c003"], "cited": ["art00-en-c003"]}
2026-10-16 11:08:08.266 | INFO     | rag.serving.pipeline:ask:75 - {"user": "root", "conversation_id": "0fd47582a2b54945baf3b93268708e23", "query": "Какой актер выполнял сурдоперевод государственного гимна на Суперкубке 50?", "status": "answered", "chunks": ["art00-ru-c003", "art00-en-c003", "art22-ru-c002", "art34-en-c004", "art06-ru-c003", "art00-en-c002", "art34-ru-c004", "art00-ru-c001"], "cited": ["art00-ru-c003", "art00-en-c003"]}
2026-10-16 11:08:08.475 | INFO     | rag.serving.pipeline:ask:75 - {"user": "root", "conversation_id": "d277f5c47d6441fcbdfbf70202a29a33", "

60 questions -> /home/alex/Courses/QA/tests/groundedness.jsonl: answered 54, not_found 6
